# Chunking Strategies Compared

**Goal:** Experiment with fixed-size, overlapping, and semantic chunking strategies — and see how each affects retrieval quality. By the end, you'll understand why chunking strategy is the #1 lever in any RAG pipeline.

**No API keys required** — this notebook uses open-source models that run locally.

---

## What You'll Learn

1. How different chunking strategies split the same document
2. Why chunk boundaries matter for retrieval quality
3. The trade-off between chunk size and retrieval precision
4. How to pick the right strategy for your use case

## Setup

In [ ]:
# Uncomment to install dependencies
# !pip install sentence-transformers scikit-learn matplotlib numpy tiktoken

import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import tiktoken
import warnings
warnings.filterwarnings('ignore')

# Load embedding model and tokenizer
model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = tiktoken.get_encoding('cl100k_base')

print(f"✓ Embedding model loaded (384 dimensions)")
print(f"✓ Tokenizer loaded (cl100k_base)")
print(f"\nSetup complete!")

## Step 1: The Source Document

We need a realistic document to chunk. This simulates a consulting firm's quarterly performance report — the kind of document that would be ingested into a RAG pipeline for the C-suite dashboard.

In [ ]:
source_document = """Q3 2025 Quarterly Performance Report

Executive Summary

The firm delivered solid Q3 results despite a seasonal slowdown in project starts. Total revenue reached $38.4 million, a 14% increase year-over-year but a 5% decrease from Q2's $40.5 million. The decline from Q2 is attributed primarily to the conclusion of two large contracts and seasonal patterns in the risk and advisory practice. Gross margin improved to 37.2%, up from 35.1% in Q2, reflecting higher utilization of senior consultants and a favorable project mix.

Financial Performance by Business Unit

Digital & Cloud led all units with $14.1 million in Q3 revenue, representing 37% of total revenue and a 16% increase year-over-year. The practice secured a $6.5 million multi-year federal technology modernization contract, which will provide a stable revenue base through 2027. The contract includes provisions for both advisory services and technology implementation, positioning the practice for sustained growth.

Risk & Advisory generated $11.2 million (29% of total), with 6% YoY growth. The practice experienced seasonal softness typical of Q3 as several banking clients delayed project starts until after their earnings season. However, the pipeline is strong with $28 million in qualified opportunities for Q4 and Q1 2026. A notable win was a $2.8 million regulatory compliance engagement with a mid-size financial institution.

Healthcare practice revenue reached $7.8 million (20% of total), growing 12% year-over-year. Growth was driven by increased demand for public health system modernization and health IT consulting. The practice secured a $2.1 million state agency digital transformation contract. Staff utilization in Healthcare was the highest across all practices at 84%.

Energy & Sustainability contributed $5.3 million (14% of total), with modest 4% YoY growth. The practice is investing in building capabilities around grid modernization and clean energy advisory, which are expected to drive accelerated growth starting in Q1 2026.

People and Talent

Total headcount reached 620 employees, a net increase of 22 from Q2. The firm hired 32 new consultants during the quarter with an offer acceptance rate of 87%. Voluntary attrition fell to 9% annualized, down from 12% in Q3 2024, reflecting improved employee engagement and competitive compensation adjustments made earlier in the year.

Consultant utilization averaged 78% in Q3, down from 82% in Q2. The decline reflects the onboarding ramp for 32 new hires and seasonal slowdown in project starts. Target utilization for Q4 is 80%. The distributed team grew to 35 members, supporting 10 active engagements across Digital & Cloud and Risk & Advisory practices.

Employee engagement score held steady at 4.1 out of 5.0. Manager effectiveness (4.4) and work-life balance (4.2) remain areas of strength. Career development clarity (3.6) and compensation competitiveness (3.5) are identified as improvement areas for the Q4 talent strategy.

Client and Business Development

The firm onboarded 10 new clients in Q3, bringing the active client count to 78. Client retention rate remains strong at 91%. Client satisfaction scores averaged 4.5 out of 5.0 across all active engagements, the highest quarterly score in firm history. Net Promoter Score improved to 58, up from 54 in Q2.

Average deal size decreased to $320K from $485K in Q2, reflecting a strategic shift toward smaller, faster-start engagements that build relationships for larger follow-on work. This approach is validated by expansion revenue: 6 existing clients expanded their engagements in Q3, contributing $11.8 million in expansion revenue, which represents 31% of total Q3 revenue.

The pipeline contains 38 qualified opportunities worth $72 million. The firm's backlog stands at $98 million, providing approximately 8 months of revenue visibility. New bookings in Q3 totaled $32 million with a book-to-bill ratio of 0.88.

Technology and AI Initiatives

The internal AI analytics platform is now used by 68% of senior leadership, up from 41% in Q2. The GenAI dashboard processes an average of 280 queries per day with a 94% user satisfaction rate. Most common query types are revenue breakdown (28%), utilization analysis (22%), client pipeline status (19%), and headcount trends (15%).

Azure cloud migration is 85% complete across all business units. Remaining workloads in finance and HR legacy systems are scheduled for Q4 completion. Cloud infrastructure costs are tracking 15% below budget due to effective resource management.

AI-related client revenue reached $5.9 million in Q3 (15% of total revenue), up from $2.8 million in Q3 2024 — representing 111% year-over-year growth. Three new AI-focused engagements launched in Q3, including the federal technology modernization contract. FedRAMP authorization was maintained for all cloud services with zero security incidents and a clean SOC 2 Type II audit."""

token_count = len(tokenizer.encode(source_document))
print(f"📄 Document: Q3 2025 Quarterly Performance Report")
print(f"   Characters: {len(source_document):,}")
print(f"   Tokens: {token_count:,}")
print(f"   Paragraphs: {len([p for p in source_document.split(chr(10)+chr(10)) if p.strip()])}")

## Step 2: Fixed-Size Chunking

The simplest approach: split text into chunks of N tokens. Fast, predictable, but blind to document structure — it will split sentences and paragraphs mid-thought.

In [ ]:
def fixed_size_chunk(text, chunk_size=200, overlap=0):
    """Split text into fixed-size token chunks with optional overlap."""
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0
    
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text.strip())
        start += chunk_size - overlap
    
    return chunks


# Create fixed-size chunks (no overlap)
fixed_chunks = fixed_size_chunk(source_document, chunk_size=200, overlap=0)

print(f"Fixed-size chunking (200 tokens, no overlap)")
print(f"{'='*60}")
print(f"Total chunks: {len(fixed_chunks)}\n")

for i, chunk in enumerate(fixed_chunks):
    tokens = len(tokenizer.encode(chunk))
    preview = chunk[:120].replace('\n', ' ')
    print(f"  Chunk {i+1} ({tokens} tokens): {preview}...")

## Step 3: Overlapping Chunking

Same as fixed-size, but chunks overlap by a percentage. This helps catch information that falls at chunk boundaries — a sentence that gets split across two chunks will appear fully in at least one of the overlapping versions.

In [ ]:
# Create overlapping chunks (20% overlap)
overlap_chunks = fixed_size_chunk(source_document, chunk_size=200, overlap=40)

print(f"Overlapping chunking (200 tokens, 40-token overlap)")
print(f"{'='*60}")
print(f"Total chunks: {len(overlap_chunks)} (vs {len(fixed_chunks)} without overlap)\n")

for i, chunk in enumerate(overlap_chunks):
    tokens = len(tokenizer.encode(chunk))
    preview = chunk[:120].replace('\n', ' ')
    print(f"  Chunk {i+1} ({tokens} tokens): {preview}...")

print(f"\n💡 Overlap added {len(overlap_chunks) - len(fixed_chunks)} extra chunks.")
print(f"   That's {(len(overlap_chunks)/len(fixed_chunks) - 1)*100:.0f}% more storage, but better boundary coverage.")

## Step 4: Semantic Chunking

Split at natural document boundaries — paragraphs, section headers. This preserves the author's intended structure and keeps related information together. Small paragraphs get merged with their neighbors to avoid tiny, meaningless chunks.

In [ ]:
def semantic_chunk(text, min_tokens=80):
    """Split text at paragraph boundaries, merging small paragraphs."""
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks = []
    current_chunk = ""
    
    for para in paragraphs:
        combined = (current_chunk + "\n\n" + para).strip() if current_chunk else para
        combined_tokens = len(tokenizer.encode(combined))
        
        if combined_tokens < min_tokens:
            current_chunk = combined
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = para
    
    if current_chunk:
        chunks.append(current_chunk)
    
    return chunks


# Create semantic chunks
sem_chunks = semantic_chunk(source_document, min_tokens=80)

print(f"Semantic chunking (paragraph boundaries, min 80 tokens)")
print(f"{'='*60}")
print(f"Total chunks: {len(sem_chunks)}\n")

for i, chunk in enumerate(sem_chunks):
    tokens = len(tokenizer.encode(chunk))
    preview = chunk[:120].replace('\n', ' ')
    print(f"  Chunk {i+1} ({tokens} tokens): {preview}...")

print(f"\n💡 Notice: chunk sizes vary because they follow document structure.")
print(f"   Each chunk contains a complete thought or section.")

## Step 5: Compare Retrieval Quality

Now the real test — which chunking strategy retrieves the most relevant context for a given question? We'll embed all chunk sets and run the same queries against each.

In [ ]:
# Embed all chunk sets
fixed_embeddings = model.encode(fixed_chunks)
overlap_embeddings = model.encode(overlap_chunks)
sem_embeddings = model.encode(sem_chunks)

# Test queries spanning different topics
test_queries = [
    "What was Q3 revenue by business unit?",
    "How is employee retention and attrition?",
    "What's the status of the federal technology contract?",
    "How are client satisfaction scores trending?",
    "What is the book-to-bill ratio?",
]

strategies = {
    'Fixed-size': (fixed_chunks, fixed_embeddings),
    'Overlapping': (overlap_chunks, overlap_embeddings),
    'Semantic': (sem_chunks, sem_embeddings),
}


def retrieve_top(query, chunks, embeddings, top_k=1):
    """Retrieve the top-k chunks for a query."""
    query_emb = model.encode([query])
    sims = cosine_similarity(query_emb, embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(chunks[i], float(sims[i])) for i in top_idx]


# Compare retrieval across strategies
for query in test_queries:
    print(f"\n🔎 Query: \"{query}\"")
    print(f"{'-'*70}")
    
    for name, (chunks, embeddings) in strategies.items():
        results = retrieve_top(query, chunks, embeddings, top_k=1)
        text, score = results[0]
        preview = text[:100].replace('\n', ' ')
        print(f"  {name:12s} (sim: {score:.4f}): {preview}...")
    
    print()

## Step 6: Visualize Retrieval Quality

Let's plot the average similarity scores across all queries for each strategy to see the overall pattern.

In [ ]:
# Compute average similarity per strategy per query
strategy_names = list(strategies.keys())
query_scores = {name: [] for name in strategy_names}

for query in test_queries:
    for name, (chunks, embeddings) in strategies.items():
        results = retrieve_top(query, chunks, embeddings, top_k=1)
        query_scores[name].append(results[0][1])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-query comparison
x = np.arange(len(test_queries))
width = 0.25
colors = ['#3498DB', '#E67E22', '#2ECC71']

for i, name in enumerate(strategy_names):
    axes[0].bar(x + i * width, query_scores[name], width, label=name, color=colors[i])

axes[0].set_xlabel('Query')
axes[0].set_ylabel('Cosine Similarity (higher = better)')
axes[0].set_title('Retrieval Quality by Query', fontweight='bold')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels([f'Q{i+1}' for i in range(len(test_queries))], fontsize=10)
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].grid(axis='y', alpha=0.3)

# Average comparison
averages = [np.mean(query_scores[name]) for name in strategy_names]
bars = axes[1].bar(strategy_names, averages, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_ylabel('Average Cosine Similarity')
axes[1].set_title('Average Retrieval Quality', fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].grid(axis='y', alpha=0.3)

for bar, avg in zip(bars, averages):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{avg:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Step 7: The Boundary Problem

The biggest risk with fixed-size chunking: it can split a critical piece of information across two chunks, so **neither chunk** captures the full meaning. Let's find a concrete example.

In [ ]:
# Find chunk boundaries in fixed-size chunking
print("The Boundary Problem: Fixed-Size Chunking")
print("=" * 60)

# Show the end of one chunk and start of the next
for i in range(len(fixed_chunks) - 1):
    end_of_chunk = fixed_chunks[i][-150:].strip()
    start_of_next = fixed_chunks[i+1][:150].strip()
    
    # Check if a sentence appears to be split
    if not fixed_chunks[i].rstrip().endswith(('.', ':', '\n')):
        print(f"\n⚠️  Split found between Chunk {i+1} and Chunk {i+2}:")
        print(f"    End of Chunk {i+1}:   \"...{end_of_chunk[-80:]}\"")
        print(f"    Start of Chunk {i+2}: \"{start_of_next[:80:]}...\"")
        print(f"    → The thought is split across two chunks!")

print(f"\n{'='*60}")
print(f"\n✅ Semantic chunking avoids this entirely by splitting at paragraph")
print(f"   boundaries where the author naturally separated topics.")

## Step 8: Chunk Size Sweet Spot

Is bigger always better? Let's test multiple chunk sizes on a single query and see how retrieval quality changes.

In [ ]:
# Test different chunk sizes
chunk_sizes = [50, 100, 150, 200, 300, 500]
test_query = "What was Q3 revenue by business unit?"
scores_by_size = []

print(f"🔎 Query: \"{test_query}\"\n")
print(f"{'Chunk Size':>12} {'Num Chunks':>12} {'Top Similarity':>16} {'Top Result Preview'}")
print(f"{'-'*12:>12} {'-'*12:>12} {'-'*16:>16} {'-'*40}")

for size in chunk_sizes:
    chunks = fixed_size_chunk(source_document, chunk_size=size, overlap=0)
    embs = model.encode(chunks)
    results = retrieve_top(test_query, chunks, embs, top_k=1)
    text, score = results[0]
    scores_by_size.append(score)
    preview = text[:40].replace('\n', ' ')
    print(f"{size:>12} {len(chunks):>12} {score:>16.4f} {preview}...")

# Plot the sweet spot curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(chunk_sizes, scores_by_size, 'o-', color='#2E86C1', linewidth=2, markersize=8)
ax.fill_between(chunk_sizes, scores_by_size, alpha=0.1, color='#2E86C1')
ax.set_xlabel('Chunk Size (tokens)', fontsize=12)
ax.set_ylabel('Top-1 Cosine Similarity', fontsize=12)
ax.set_title('Retrieval Quality vs. Chunk Size', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Annotate the sweet spot
best_idx = np.argmax(scores_by_size)
ax.annotate(f'Best: {chunk_sizes[best_idx]} tokens',
           xy=(chunk_sizes[best_idx], scores_by_size[best_idx]),
           xytext=(chunk_sizes[best_idx] + 50, scores_by_size[best_idx] - 0.02),
           arrowprops=dict(arrowstyle='->', color='#E74C3C'),
           fontsize=11, color='#E74C3C', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n🎯 Takeaway: Too small → chunks lack context. Too large → chunks dilute the signal.")
print(f"   The sweet spot for most enterprise documents is 150-300 tokens.")

## Key Takeaways

1. **Semantic chunking preserves meaning.** By splitting at natural boundaries, each chunk contains a complete thought. Fixed-size chunking is simpler but risks splitting critical information.

2. **Overlap is cheap insurance.** A 20% overlap adds ~25% more chunks (storage cost) but catches information that falls at boundaries. Always worth it for fixed-size strategies.

3. **Chunk size has a sweet spot.** Too small (< 100 tokens) and chunks lack context. Too large (> 500 tokens) and the embedding captures too many topics at once, reducing precision. Aim for 150-300 tokens.

4. **The boundary problem is real.** When critical information gets split across two chunks, neither chunk captures it well. This is RAG's silent failure mode — the answer exists in your data, but chunking made it invisible.

5. **Strategy depends on your documents.** Well-structured reports → semantic chunking. Unstructured text dumps → fixed-size with overlap. Mixed formats → hierarchical (parent-child) chunking.

| Strategy | Best For | Avoid When |
|---|---|---|
| **Fixed-size** | Quick prototypes, uniform text | Documents with clear section structure |
| **Overlapping** | Any fixed-size use case | Storage is extremely constrained |
| **Semantic** | Reports, articles, structured docs | Unstructured text without paragraphs |

---

### Next Steps
- **[04-embeddings-explorer.ipynb](04-embeddings-explorer.ipynb)** — Visualize how embeddings capture meaning in 2D/3D space
- **[02-build-a-rag-pipeline.ipynb](02-build-a-rag-pipeline.ipynb)** — Build a full RAG system using these chunking strategies
- **[RAG Deep Dive](../docs/04-rag-deep-dive.md)** — The complete guide to RAG pipeline engineering